# 👨‍🍳 Azure Language Recipes

These recipes cover language detection, prebuilt and custom named entity recognition, PII detection, and Text Analytics for health. Features announced for retirement are excluded.

The notebook uses `azure-ai-textanalytics==6.0.0b2` and explicitly selects the stable service API `2025-11-01`, the newest stable version documented for that SDK. The SDK package itself is preview; its default preview API is not used.

## Creating the client: `TextAnalysisClient`

Locally, sign in with `azd auth login` and launch this notebook through `azd exec` as described in the README.

On an Azure host with managed identity enabled, set `AZURE_AUTH_MODE=managed_identity` and `AZURE_LANGUAGE_ENDPOINT`. For a user-assigned managed identity, also set `AZURE_CLIENT_ID` to its client ID; leave it unset for the host's system-assigned identity. Grant the calling identity the **Cognitive Services Language Reader** role on the Language resource. Managed identity does not work on an ordinary local laptop.

In [ ]:
import json
import os

from azure.ai.textanalytics import TextAnalysisClient, models
from azure.identity import AzureDeveloperCliCredential, ManagedIdentityCredential

API_VERSION = "2025-11-01"
endpoint = os.environ["AZURE_LANGUAGE_ENDPOINT"]
auth_mode = os.environ.get("AZURE_AUTH_MODE", "azd")

if auth_mode == "managed_identity":
    credential = ManagedIdentityCredential(client_id=os.environ.get("AZURE_CLIENT_ID"))
elif auth_mode == "azd":
    credential = AzureDeveloperCliCredential(tenant_id=os.environ["AZURE_TENANT_ID"])
else:
    raise ValueError("AZURE_AUTH_MODE must be 'azd' or 'managed_identity'.")

client = TextAnalysisClient(
    endpoint=endpoint,
    credential=credential,
    api_version=API_VERSION,
)


def show_result(result: models.AnalyzeTextResult | models.TextActions) -> None:
    payload = result.as_dict()
    print(json.dumps(payload, indent=2, ensure_ascii=False))
    if isinstance(result, models.TextActions):
        if result.failed:
            raise RuntimeError("A Language task failed; see the result above.")
        items = payload.get("items")
        if not items:
            raise RuntimeError("The completed job returned no task results.")
    else:
        items = [payload]
    for item in items:
        if item.get("status") in {"failed", "cancelled"}:
            raise RuntimeError("A Language task did not succeed; see the result above.")
        results = item.get("results")
        if results is None:
            raise RuntimeError("The service response is missing analysis results.")
        if results.get("errors"):
            raise RuntimeError("One or more documents failed; see the result above.")


print(f"Service API: {API_VERSION}; authentication: {auth_mode}")

## Detecting language: `analyze_text`

Returns the detected language, ISO code, confidence score, warnings, and statistics for each document.

In [ ]:
result = client.analyze_text(
    models.TextLanguageDetectionInput(
        text_input=models.LanguageDetectionTextInput(
            language_inputs=[
                models.LanguageInput(id="1", text="Ce document est rédigé en français.", country_hint="FR")
            ]
        ),
        action_content=models.LanguageDetectionActionContent(
            logging_opt_out=True,
            model_version="latest",
        ),
    ),
    show_stats=True,
)
show_result(result)

## Recognising named entities: `analyze_text`

Returns entity categories, confidence scores, and Unicode code-point offsets from a prebuilt NER model.

In [ ]:
result = client.analyze_text(
    models.TextEntityRecognitionInput(
        text_input=models.MultiLanguageTextInput(
            multi_language_inputs=[
                models.MultiLanguageInput(
                    id="1",
                    language="en",
                    text="Contoso opened a new office in Seattle on March 3, 2024, hiring 200 engineers.",
                )
            ]
        ),
        action_content=models.EntitiesActionContent(
            logging_opt_out=True,
            model_version="latest",
            string_index_type="UnicodeCodePoint",
        ),
    ),
    show_stats=True,
)
show_result(result)

## Detecting personal information (PII): `analyze_text`

Uses synthetic sample data. Returns matching PII entities and redacted text; service-side input logging is disabled.

In [ ]:
result = client.analyze_text(
    models.TextPiiEntitiesRecognitionInput(
        text_input=models.MultiLanguageTextInput(
            multi_language_inputs=[
                models.MultiLanguageInput(
                    id="1",
                    language="en",
                    text="The employee's SSN is 859-98-0987 and her phone number is 555-555-5555.",
                )
            ]
        ),
        action_content=models.PiiActionContent(
            pii_categories=["USSocialSecurityNumber", "PhoneNumber"],
            logging_opt_out=True,
            model_version="latest",
            string_index_type="UnicodeCodePoint",
        ),
    ),
    show_stats=True,
)
show_result(result)

## Running multiple analyses in one job: `begin_analyze_text_job`

Combines only non-retiring features: prebuilt NER and PII detection. The SDK handles polling and pagination; each returned `TextActions` page contains its task results.

In [ ]:
poller = client.begin_analyze_text_job(
    text_input=models.MultiLanguageTextInput(
        multi_language_inputs=[
            models.MultiLanguageInput(
                id="1",
                language="en",
                text="Contact Jane Smith at jane.smith@example.com about Contoso's Seattle office.",
            )
        ]
    ),
    actions=[
        models.EntitiesLROTask(
            name="entities",
            parameters=models.EntitiesActionContent(
                logging_opt_out=True,
                model_version="latest",
                string_index_type="UnicodeCodePoint",
            ),
        ),
        models.PiiLROTask(
            name="pii",
            parameters=models.PiiActionContent(
                logging_opt_out=True,
                model_version="latest",
                string_index_type="UnicodeCodePoint",
            ),
        ),
    ],
    display_name="entities-and-pii",
    polling_interval=5,
)
for page in poller.result():
    show_result(page)
print("Job details:", poller.details)

## Analysing healthcare entities: `begin_analyze_text_job`

Uses synthetic clinical text. Results include healthcare entities, normalized terms, assertions, data-source links, and relationships.

In [ ]:
poller = client.begin_analyze_text_job(
    text_input=models.MultiLanguageTextInput(
        multi_language_inputs=[
            models.MultiLanguageInput(
                id="1",
                language="en",
                text="The patient was prescribed 100 mg of ibuprofen twice daily for knee pain.",
            )
        ]
    ),
    actions=[
        models.HealthcareLROTask(
            name="healthcare",
            parameters=models.HealthcareTaskParameters(
                logging_opt_out=True,
                model_version="latest",
                string_index_type="UnicodeCodePoint",
            ),
        )
    ],
    display_name="healthcare-entities",
    polling_interval=5,
)
for page in poller.result():
    show_result(page)
print("Job details:", poller.details)

## Recognising custom entities: `begin_analyze_text_job` (optional)

Custom NER is not custom text classification: custom NER remains in the core feature set. This recipe requires an existing, trained and deployed custom NER model on the Language resource.

Set `AZURE_LANGUAGE_CUSTOM_NER_PROJECT` and `AZURE_LANGUAGE_CUSTOM_NER_DEPLOYMENT` before launching the notebook, or skip this cell. The Bicep template does not create training storage, projects, or models.

In [ ]:
project_name = os.environ["AZURE_LANGUAGE_CUSTOM_NER_PROJECT"]
deployment_name = os.environ["AZURE_LANGUAGE_CUSTOM_NER_DEPLOYMENT"]

poller = client.begin_analyze_text_job(
    text_input=models.MultiLanguageTextInput(
        multi_language_inputs=[
            models.MultiLanguageInput(
                id="1",
                language="en",
                text="The loan agreement between John Smith and Contoso Bank was signed for $250,000.",
            )
        ]
    ),
    actions=[
        models.CustomEntitiesLROTask(
            name="custom-entities",
            parameters=models.CustomEntitiesActionContent(
                project_name=project_name,
                deployment_name=deployment_name,
                logging_opt_out=True,
                string_index_type="UnicodeCodePoint",
            ),
        )
    ],
    display_name="custom-entities",
    polling_interval=5,
)
for page in poller.result():
    show_result(page)
print("Job details:", poller.details)

## Closing the client and credential

Run this last. Rerun the setup cell before starting another recipe.

In [ ]:
client.close()
credential.close()